# M4 exploration: does wind change airborne traffic near Turkish airports?

**Reading only.** Every number comes from the M3 clean tables through the saved queries in
`sql/queries/02-04` and the helpers in `src/explore.py`. The database is opened read-only and
fingerprinted before and after (last cell): if any table changed, the notebook fails.

**Question (from the README):** do stronger gusts go together with more or faster traffic
near an airport?

**What the data can and cannot say.** There are only a handful of ADS-B snapshots, taken on
irregular days. The honest reading of any result below is *exploratory*: it shows what the
data looks like and which question a larger sample could answer.

**Decisions fixed before looking at any traffic outcome**

| Decision | Value | Why |
|---|---|---|
| Airport area | 0.25 deg, about 28 km, nearest airport wins | M2 rule, now without double counting |
| Gust filter | `ok` only (sensitivity: `ok` + `unusual_high`) | `unusual_high` is extreme but possible, so we show both |
| Forecast hours | excluded (invariant I-20) | a forecast is not an observation |
| Clock | UTC in the database, `Europe/Istanbul` (UTC+3) for every day or hour shown | people fly by local time |
| "Windy" | gust at or above the airport's own 75th percentile of trusted hours (sensitivity: median) | airports differ: a 17 km/h gust is the strongest hour at Ankara but an ordinary one at Istanbul |
| Outcome | airborne aircraft per snapshot, and mean airborne speed, each minus that airport's mean | removes the airport-size effect |
| Independent unit | local calendar day | snapshots on the same day are not independent |
| Local-day coverage rule | at least 18 of 24 hours with a trusted gust | 75% |

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "explore.py").exists())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import explore

# --- every parameter of the analysis, in one place ---
RADIUS_DEG = 0.25
LON_SCALE = 0.76                      # cos(40.5 deg): one degree of longitude is shorter than one of latitude
GUST_FLAGS = ("ok",)
GUST_FLAGS_WIDE = ("ok", "unusual_high")
MIN_SHARE = 0.75                      # share of the 24 local hours that must carry a trusted gust
WINDY_Q, WINDY_Q_ALT = 0.75, 0.50     # per-airport quantile that defines "windy"
MIN_UNITS = 8                         # fewer independent days than this: exploratory only
N_BOOT = int(os.environ.get("M4_N_BOOT", 5000))
SEED = 20261103
TRAFFIC_PARAMS = {"radius_deg": RADIUS_DEG, "lon_scale": LON_SCALE}
CLOCK = "clock: Europe/Istanbul (UTC+3)"

con = explore.connect_ro()
fp_before = explore.db_fingerprint(con)
print("database opened read-only; fingerprint taken for", len(fp_before), "tables")

OperationalError: no such table: aircraft

## 1. Weather supply: how many trusted hours exist?

Query `03` returns every observed hour (forecasts removed) and keeps rows whose gust is not
trusted, with the value set to NULL, so the denominators stay visible. We then count per
**local** day, because a UTC day cuts a Turkish day in two (the UTC table from query `04` is
kept as the data-supply view).

In [ ]:
w = explore.query_df(con, "03_weather_hours_clean", explore.flag_params(GUST_FLAGS))
w["local_time"] = explore.to_local(w.observed_at)
w["local_date"] = w.local_time.dt.strftime("%Y-%m-%d")

cov_utc = explore.query_df(con, "04_coverage_weather", explore.flag_params(GUST_FLAGS))
cov_local = explore.coverage_table(w, ["iata", "local_date"], "wind_gusts_10m",
                                   expected=24, min_share=MIN_SHARE)

print(f"{len(w)} observed weather hours, {int(w.gust_usable.sum())} with a trusted gust "
      f"({w.gust_usable.mean():.1%}); forecast hours already removed")
print(f"UTC days with data: {cov_utc.utc_date.nunique()}; local days with data: {cov_local.local_date.nunique()}")
summary = (cov_local.groupby("iata")
           .agg(days=("local_date", "size"), days_meeting_rule=("meets_rule", "sum"),
                trusted_hours=("n", "sum")))
summary

In [ ]:
wide = explore.query_df(con, "03_weather_hours_clean", explore.flag_params(GUST_FLAGS_WIDE))
sens = pd.DataFrame({
    "hours": w.groupby("iata").size(),
    "gust_ok": w.groupby("iata").gust_usable.sum(),
    "gust_ok_or_unusual_high": wide.groupby("iata").gust_usable.sum(),
})
sens["extra_hours_kept_by_wide_filter"] = sens.gust_ok_or_unusual_high - sens.gust_ok
sens["max_gust_ok"] = w.groupby("iata").wind_gusts_10m.max()
sens["max_gust_wide"] = wide.groupby("iata").wind_gusts_10m.max()
sens

In [ ]:
dates = [d.strftime("%Y-%m-%d") for d in
         pd.date_range(cov_local.local_date.min(), cov_local.local_date.max(), freq="D")]
airports = sorted(cov_local.iata.unique())
grid = (cov_local.pivot(index="iata", columns="local_date", values="share")
        .reindex(index=airports, columns=dates))

fig, ax = plt.subplots(figsize=(11, 3.2))
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("#e6e6e6")               # days with no weather row at all
im = ax.imshow(np.ma.masked_invalid(grid.to_numpy(dtype=float)), aspect="auto", vmin=0, vmax=1, cmap=cmap)
ax.set_yticks(range(len(airports)), airports)
ax.set_xticks(range(len(dates)), [d[5:] for d in dates], rotation=90, fontsize=7)
ax.set_title("Share of the 24 local hours with a trusted gust (grey = no data that day)")
fig.colorbar(im, ax=ax, label="share")
explore.save_fig(fig, "fig01_weather_coverage",
                 f"n = {int(w.gust_usable.sum())} trusted hours of {len(w)}; flag filter: {', '.join(GUST_FLAGS)}; {CLOCK}")
plt.show()

## 2. Traffic grid: how much was airborne near each airport?

Query `02` gives **one row per snapshot and airport, zeros included**. A state in the overlap of
two airport areas is counted once, for the nearer airport. Before using it we check that the
grid has as many snapshots as there are landed OpenSky files: a snapshot whose states were all
repeats of earlier ones would be missing from the clean table, and with it its zeros.

In [ ]:
traffic = explore.query_df(con, "02_snapshot_traffic", TRAFFIC_PARAMS)
traffic["local_time"] = explore.to_local(traffic.snapshot)
traffic["local_date"] = traffic.local_time.dt.strftime("%Y-%m-%d")
traffic["local_hour"] = traffic.local_time.dt.hour + traffic.local_time.dt.minute / 60

n_snap = traffic.snapshot.nunique()
raw_dir = ROOT / "data" / "raw" / "opensky"
raw_files = sorted(raw_dir.glob("*.json")) if raw_dir.exists() else []
if raw_files:
    status = "OK" if len(raw_files) == n_snap else "MISMATCH: check for snapshots lost in M2/M3 deduplication"
    print(f"snapshots in grid: {n_snap}; landed OpenSky files: {len(raw_files)} -> {status}")
else:
    print(f"snapshots in grid: {n_snap}; raw files not available here, snapshot completeness not checked")

print(f"{len(traffic)} grid rows = {n_snap} snapshots x {traffic.iata.nunique()} airports; "
      f"{(traffic.n_aircraft == 0).sum()} rows are real zeros")
print(f"local days with at least one snapshot: {traffic.local_date.nunique()}")
per_airport = traffic.groupby("iata").agg(
    snapshots=("snapshot", "size"), mean_airborne=("n_airborne", "mean"),
    max_airborne=("n_airborne", "max"), mean_ground=("n_ground", "mean"))
per_airport.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for iata, part in traffic.groupby("iata"):
    ax.scatter(part.local_hour, part.n_airborne, label=iata, alpha=0.8)
ax.set_xlim(0, 24)
ax.set_xlabel("local hour of the snapshot")
ax.set_ylabel("airborne aircraft within 28 km")
ax.set_title("Airborne traffic by local time of day (one point per snapshot and airport)")
ax.legend(title="airport", ncols=5, fontsize=8)
explore.save_fig(fig, "fig02_traffic_by_local_hour",
                 f"n = {len(traffic)} snapshot-airport rows from {n_snap} snapshots; radius {RADIUS_DEG} deg; {CLOCK}")
plt.show()

## 3. Joining traffic and weather

The join is a LEFT join from the traffic grid on `(airport, UTC hour)`; a snapshot at 10:59 uses
the weather of hour 10:00. Rows without a trusted gust stay in the table (`gust_usable = 0`) and
are counted below, never dropped silently.

In [ ]:
joined = explore.join_traffic_weather(traffic, w.drop(columns=["local_time", "local_date"]))
usable = joined[joined.gust_usable == 1]
print(f"{len(joined)} grid rows; {int(joined.has_weather.sum())} have a weather hour; "
      f"{len(usable)} have a trusted gust ({len(usable) / len(joined):.0%})")
print("rows lost to missing or untrusted gusts, by airport:")
print((joined.assign(lost=joined.gust_usable == 0).groupby("iata").lost.sum()).to_string())

## 4. Windy versus calm

For each airport the cut-off is its own 75th percentile of trusted gusts (median as a check).
The outcome is the airborne count and the mean airborne speed **minus the airport's mean**, so
Istanbul's larger traffic does not masquerade as a wind effect. The anomaly is computed on all
grid rows **before** rows are filtered by weather. The interval comes from resampling whole
local days (`bootstrap_diff`), and every line states how many days stand behind it.

In [ ]:
anom = explore.add_airport_anomaly(joined, ["n_airborne", "mean_velocity_airborne"])
OUTCOMES = {"n_airborne_anom": "airborne aircraft (vs airport mean)",
            "mean_velocity_airborne_anom": "mean airborne speed, m/s (vs airport mean)"}

def windy_calm(frame, q):
    cut = explore.gust_cutoffs(w, q)
    labelled = explore.split_by_threshold(frame, "wind_gusts_10m", cut)
    return labelled.dropna(subset=["group"]), cut

rows = []
for q in (WINDY_Q, WINDY_Q_ALT):
    part, cut = windy_calm(anom, q)
    counts = part.group.value_counts()
    for outcome in OUTCOMES:
        result = explore.bootstrap_diff(part, "local_date", outcome, "group", n_boot=N_BOOT, seed=SEED)
        rows.append({"windy_quantile": q, "outcome": outcome,
                     "n_calm": int(counts.get("calm", 0)), "n_windy": int(counts.get("windy", 0)),
                     "diff_windy_minus_calm": result[0], "lo": result[1], "hi": result[2],
                     "n_days": result[3]})
        print(f"q={q:.2f}  {OUTCOMES[outcome]}: {explore.format_diff(result, MIN_UNITS)}")
effect = pd.DataFrame(rows)
effect.round(2)

In [ ]:
# Does one airport drive the result? Repeat the main comparison leaving each airport out.
part, _ = windy_calm(anom, WINDY_Q)
loo = []
for iata in sorted(part.iata.unique()):
    sub = part[part.iata != iata]
    if sub.group.nunique() < 2:
        continue
    d, lo, hi, n = explore.bootstrap_diff(sub, "local_date", "n_airborne_anom", "group",
                                          n_boot=N_BOOT, seed=SEED)
    loo.append({"left_out": iata, "diff": d, "lo": lo, "hi": hi, "n_days": n})
loo = pd.DataFrame(loo)
loo.round(2)

In [ ]:
part, cut = windy_calm(anom, WINDY_Q)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = dict(zip(sorted(part.iata.unique()), plt.cm.tab10.colors))
for ax, outcome in zip(axes, OUTCOMES):
    for iata, p in part.groupby("iata"):
        ax.scatter(p.wind_gusts_10m, p[outcome], color=colors[iata], label=iata, alpha=0.8)
    ax.axhline(0, color="#888888", lw=0.8)
    ax.set_xlabel("wind gust at the airport, km/h")
    ax.set_ylabel(OUTCOMES[outcome])
axes[0].legend(title="airport", fontsize=8)
fig.suptitle("Traffic against gusts (each airport relative to its own mean)")
explore.save_fig(fig, "fig03_traffic_vs_gust",
                 f"n = {len(part)} snapshot-airport rows with a trusted gust; flag filter: {', '.join(GUST_FLAGS)}; {CLOCK}")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, outcome in zip(axes, OUTCOMES):
    sub = effect[effect.outcome == outcome].reset_index(drop=True)
    y = np.arange(len(sub))
    ax.errorbar(sub.diff_windy_minus_calm, y,
                xerr=[sub.diff_windy_minus_calm - sub.lo, sub.hi - sub.diff_windy_minus_calm],
                fmt="o", capsize=4, color="#1f77b4")
    ax.axvline(0, color="#888888", lw=0.8)
    ax.set_yticks(y, [f"windy = above airport {int(q * 100)}th pct" for q in sub.windy_quantile])
    ax.set_xlabel("windy minus calm, 95% interval")
    ax.set_title(OUTCOMES[outcome], fontsize=9)
fig.suptitle(f"Windy versus calm, {int(effect.n_days.max())} local days: how much the data can settle")
explore.save_fig(fig, "fig04_windy_vs_calm",
                 f"n = {int(effect.n_days.max())} local days; bootstrap of days, {N_BOOT} draws, seed {SEED}; "
                 f"flag filter: {', '.join(GUST_FLAGS)}; {CLOCK}")
plt.show()

## 5. What we can say

The text below is generated from the results above, so it cannot drift from them.

In [ ]:
main = effect[effect.windy_quantile == WINDY_Q].set_index("outcome")
n_days = int(main.n_days.iloc[0])
for outcome, label in OUTCOMES.items():
    r = main.loc[outcome]
    excludes = not (r.lo <= 0 <= r.hi)
    print(f"- {label}: windy minus calm = {r.diff_windy_minus_calm:+.2f} "
          f"[{r.lo:+.2f}, {r.hi:+.2f}], {int(r.n_windy)} windy vs {int(r.n_calm)} calm rows, "
          f"{n_days} local days -> {'interval excludes 0' if excludes else 'no detectable difference'}.")
if n_days < MIN_UNITS:
    print(f"- Only {n_days} independent days: treat every line above as exploratory.")
print(f"- Snapshot timing is irregular ({n_snap} snapshots, {traffic.local_date.nunique()} local days): "
      "time of day is not controlled and confounds any wind effect.")
print("- Next step for a firmer answer: more snapshots on more days, spread over the day, "
      "and a model with time of day and airport as covariates.")

## 6. Reproducibility check

Nothing above may have changed the database. This cell must pass for the notebook to count as run.

In [ ]:
fp_after = explore.db_fingerprint(con)
explore.assert_unchanged(fp_before, fp_after)
print("OK: all", len(fp_after), "table fingerprints are identical before and after the notebook")
con.close()